# 02 — Sentiment model

Character n-gram TF-IDF + Logistic Regression. Feeds **thesis 4.6 (Algorithms/Models)**
and **5.6 (Accuracy/Precision/Recall)**.

Two jobs: classify Positive / Neutral / Negative, and produce the continuous
`P(positive) − P(negative)` score that SSG needs in step 5.


## Kernel check


In [ ]:
import sys; print(sys.executable)


## Train

Grid search runs over 54 configurations on the project's five fixed folds, then the
calibration check compares raw against Platt-scaled probabilities out-of-fold.
Takes about 30 seconds.


In [ ]:
from IPython.display import Image, display
from trustlens import config, sentiment

res = sentiment.report_all()
meta = res['meta']
meta


## Why these hyperparameters

They were searched, not chosen. Worth noting that the search independently landed on
`ngram_range=(2,4)` — the value the project had fixed by decision. That is now evidence
rather than assertion, and the grid is reportable.


In [ ]:
sentiment.table_grid(res['gs'])


## Results


In [ ]:
y_true = res['test']['sentiment_polarity'].values
sentiment.per_class_table(y_true, res['y_pred'])


In [ ]:
sentiment.per_language_table(res['test'], res['y_pred'])


In [ ]:
for n in ['07_sentiment_confusion', '08_sentiment_by_language', '09_score_vs_star']:
    display(Image(filename=str(config.FIGURES/f'{n}.png'), width=560))


## The score SSG will use

`score()` returns `P(positive) − P(negative)` on [−1, 1]. The table below compares its
mean against the star-implied value `(r−3)/2` that SSG measures the gap from.

The offsets are not noise — 2-star reviews sit about 0.30 below where the linear mapping
puts them. Carry this into step 5: part of SSG will be a constant offset per star rating
rather than genuine inconsistency, so report SSG against an empirically centred mapping
as a sensitivity check alongside the pre-specified `(r−3)/2`.


In [ ]:
scores = sentiment.score(res['model'], res['train']['review_text'].values)
sentiment.table_score_vs_star(res['train'], scores)


## Using the saved model


In [ ]:
m = sentiment.load()
examples = ['ගොඩක් හොඳයි', 'hari hodai recommend karanawa',
            'worst product, stopped working', 'ok']
for t, p, s in zip(examples, m.predict(examples), sentiment.score(m, examples)):
    print(f'{s:+.3f}  {p:<8} {t}')


## What goes where

| Output | Section |
|---|---|
| `08_sentiment_grid_search.csv` | 4.6 — model selection |
| `06_sentiment_per_class.csv`, figure `07` | 5.6 — technical measures |
| `07_sentiment_per_language.csv`, figure `08` | 5.6, and the cross-lingual discussion |
| `09_score_vs_star.csv`, figure `09` | 4.6 — CII design, and 5.4 |
